# XGBoost — Riesgo cardiovascular (BRFSS 2024)

Este cuaderno implementa el modelo XGBoost dentro del protocolo experimental descrito en la Sección IV del artículo. La primera fase valida el procedimiento sobre un único pliegue antes de ejecutar el ciclo completo de validación cruzada de 5 pliegues.

In [35]:
import sys
sys.path.append("..")

import pandas as pd
from xgboost import XGBClassifier

from scripts import pipeline as pl

DATASET_PATH = "../Dataset/brfss_2024_cvd_fairness_clean.csv"

In [36]:
df = pd.read_csv(DATASET_PATH)
df = pl.construir_target(df)

X, y, protegidas = pl.separar_X_y_protegidas(df)
folds = pl.generar_folds(X, y, n_splits=5)

print("filas:", len(df), "| columnas en X:", X.shape[1], "| proporcion riesgo:", y.mean().round(4))

filas: 449760 | columnas en X: 30 | proporcion riesgo: 0.1182


## Prueba sobre el pliegue 0

Antes de ejecutar el ciclo completo de 5 pliegues, se valida el procedimiento sobre un único pliegue, con el fin de confirmar que el modelo entrena correctamente sobre estos datos y que la función `pipeline.evaluar()` procesa sus predicciones sin errores.

In [37]:
tr_idx, te_idx = folds[0]

X_train, X_test = X.iloc[tr_idx].reset_index(drop=True), X.iloc[te_idx].reset_index(drop=True)
y_train, y_test = y.iloc[tr_idx].reset_index(drop=True), y.iloc[te_idx].reset_index(drop=True)
prot_test = protegidas.iloc[te_idx].reset_index(drop=True)

X_train, X_test = pl.imputar_proporcional(X_train, X_test)
X_train, X_test = pl.normalizar(X_train, X_test)
X_train, y_train = pl.aplicar_smote(X_train, y_train)

print("train tras SMOTE:", y_train.value_counts().to_dict())

train tras SMOTE: {0: 317268, 1: 317268}


### Tratamiento de variables categóricas

Las variables categóricas (`_RACE`, `_INCOMG1`, entre otras) se mantienen codificadas como valores numéricos, sin asignarles el tipo `category` nativo de XGBoost ni aplicar codificación *one-hot*. XGBoost admite un tratamiento nativo de variables categóricas que podría mejorar su desempeño individual, pero dicho tratamiento no está disponible de forma equivalente en Regresión Logística ni en SVM. Dado que el objetivo del estudio es comparar los cuatro modelos bajo condiciones de preprocesamiento idénticas, se descarta ese tratamiento diferenciado para preservar la validez de la comparación.

In [38]:
modelo = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    tree_method="hist",
    random_state=pl.RANDOM_STATE,
    eval_metric="logloss",
)
modelo.fit(X_train, y_train)

y_pred = modelo.predict(X_test)
y_proba = modelo.predict_proba(X_test)[:, 1]

In [39]:
resumen, detalle = pl.evaluar(y_test, y_pred, y_proba, prot_test)

print("RESUMEN (pliegue 0):")
for k, v in resumen.items():
    print(f"  {k}: {v:.4f}")

detalle["_RACE"]

RESUMEN (pliegue 0):
  auroc: 0.8271
  eod_sexo: 0.0694
  fpr_disp_sexo: 0.0116
  calib_desv_sexo: 0.0447
  eod_raza: 0.1305
  fpr_disp_raza: 0.0159
  calib_desv_raza: 0.1385
  eod_ingreso: 0.1767
  fpr_disp_ingreso: 0.0334
  calib_desv_ingreso: 0.0740


,grupo,n,tpr,fpr,calibracion
0,1.0,65317,0.129879,0.014069,1.040381
1,2.0,6858,0.084677,0.013085,1.036103
2,4.0,2410,0.048077,0.002168,1.135454
3,7.0,2016,0.135802,0.014664,0.963533
4,8.0,9348,0.067358,0.003649,1.118642
5,6.0,693,0.166667,0.008292,1.019161
6,3.0,1218,0.178571,0.018095,0.988058
7,5.0,381,0.152174,0.005970,0.861540


## Ajuste de hiperparámetros

Se realiza una búsqueda en grid sobre combinaciones de `n_estimators`, `max_depth` y `learning_rate`, con el fin de justificar la elección de hiperparámetros con evidencia, en vez de utilizar valores arbitrarios.

Para no comprometer los datos de prueba de ningún pliegue (lo cual constituiría fuga de datos), la búsqueda se realiza dentro del propio train del pliegue 0: este se separa en una partición interna de entrenamiento (80 %) y una de validación (20 %), y es esta última la que se utiliza para comparar las combinaciones. Los 5 pliegues oficiales reportados en la siguiente sección permanecen intactos durante todo este proceso.

In [40]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

tr_idx_fold0, _ = folds[0]
X_train_fold0 = X.iloc[tr_idx_fold0].reset_index(drop=True)
y_train_fold0 = y.iloc[tr_idx_fold0].reset_index(drop=True)

X_tr_int, X_val_int, y_tr_int, y_val_int = train_test_split(
    X_train_fold0, y_train_fold0,
    test_size=0.2, stratify=y_train_fold0, random_state=pl.RANDOM_STATE,
)

X_tr_int, X_val_int = pl.imputar_proporcional(X_tr_int, X_val_int)
X_tr_int, X_val_int = pl.normalizar(X_tr_int, X_val_int)
X_tr_int, y_tr_int = pl.aplicar_smote(X_tr_int, y_tr_int)

print("train interno:", X_tr_int.shape, "| validacion interna:", X_val_int.shape)

train interno: (507628, 30) | validacion interna: (71962, 30)


In [41]:
grid = [
    dict(n_estimators=n_estimators, max_depth=max_depth, learning_rate=learning_rate)
    for n_estimators in [100, 300]
    for max_depth in [4, 6, 8]
    for learning_rate in [0.05, 0.1]
]

resultados_grid = []
for params in grid:
    modelo_grid = XGBClassifier(
        **params, tree_method="hist", random_state=pl.RANDOM_STATE, eval_metric="logloss",
    )
    modelo_grid.fit(X_tr_int, y_tr_int)
    proba_val = modelo_grid.predict_proba(X_val_int)[:, 1]
    resultados_grid.append({**params, "auroc_validacion": roc_auc_score(y_val_int, proba_val)})

tabla_grid = pd.DataFrame(resultados_grid).sort_values("auroc_validacion", ascending=False)
tabla_grid.round(4)

,n_estimators,max_depth,learning_rate,auroc_validacion
9,300,6,0.10,0.8281
10,300,8,0.05,0.8280
8,300,6,0.05,0.8272
7,300,4,0.10,0.8271
5,100,8,0.10,0.8268
11,300,8,0.10,0.8254
3,100,6,0.10,0.8248
4,100,8,0.05,0.8247
6,300,4,0.05,0.8227
2,100,6,0.05,0.8223


La combinación con mejor AUROC de validación es `n_estimators=300, max_depth=6, learning_rate=0.1` (0.8281), prácticamente empatada con `n_estimators=300, max_depth=8, learning_rate=0.05` (0.8280). Dado que ambas combinaciones rinden igual, se elige la de menor profundidad (6 en vez de 8), ya que un árbol menos profundo reduce el riesgo de sobreajuste sin sacrificar desempeño. Se observa además que `n_estimators=100` queda consistentemente por debajo de `n_estimators=300` en igualdad de los demás parámetros, lo que indica que 100 árboles no son suficientes para que el modelo converja.

In [42]:
MEJORES_HIPERPARAMETROS = dict(n_estimators=300, max_depth=6, learning_rate=0.1)
MEJORES_HIPERPARAMETROS

{'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.1}

## Ciclo completo de validación cruzada (5 pliegues)

Una vez confirmado el funcionamiento sobre el pliegue 0, se repite el mismo procedimiento de forma independiente para cada uno de los 5 pliegues, y se reporta el promedio y la desviación estándar de cada métrica entre pliegues, tal como se especifica en la Sección IV-F del artículo.

In [43]:
resultados = []
detalle_total = []

for i, (tr_idx, te_idx) in enumerate(folds):
    X_train, X_test = X.iloc[tr_idx].reset_index(drop=True), X.iloc[te_idx].reset_index(drop=True)
    y_train, y_test = y.iloc[tr_idx].reset_index(drop=True), y.iloc[te_idx].reset_index(drop=True)
    prot_test = protegidas.iloc[te_idx].reset_index(drop=True)

    X_train, X_test = pl.imputar_proporcional(X_train, X_test)
    X_train, X_test = pl.normalizar(X_train, X_test)
    X_train, y_train = pl.aplicar_smote(X_train, y_train)

    modelo = XGBClassifier(
        **MEJORES_HIPERPARAMETROS,
        tree_method="hist",
        random_state=pl.RANDOM_STATE,
        eval_metric="logloss",
    )
    modelo.fit(X_train, y_train)

    y_pred = modelo.predict(X_test)
    y_proba = modelo.predict_proba(X_test)[:, 1]

    resumen, detalle = pl.evaluar(y_test, y_pred, y_proba, prot_test)
    resumen["modelo"] = "xgboost"
    resumen["pliegue"] = i
    resultados.append(resumen)

    for col, tabla_grupo in detalle.items():
        tabla_grupo = tabla_grupo.copy()
        tabla_grupo["atributo"] = pl.SUFIJOS_PROTEGIDAS[col]
        tabla_grupo["modelo"] = "xgboost"
        tabla_grupo["pliegue"] = i
        detalle_total.append(tabla_grupo)

    print(f"pliegue {i} listo | AUROC = {resumen['auroc']:.4f}")

pliegue 0 listo | AUROC = 0.8271
pliegue 1 listo | AUROC = 0.8246
pliegue 2 listo | AUROC = 0.8280
pliegue 3 listo | AUROC = 0.8239
pliegue 4 listo | AUROC = 0.8268


In [44]:
tabla_metricas = pd.DataFrame(resultados)
columnas_ordenadas = ["modelo", "pliegue"] + [c for c in tabla_metricas.columns if c not in ("modelo", "pliegue")]
tabla_metricas = tabla_metricas[columnas_ordenadas]

resumen_cv = tabla_metricas.drop(columns=["modelo", "pliegue"]).agg(["mean", "std"]).round(4)

tabla_metricas.round(4)

,modelo,pliegue,auroc,eod_sexo,fpr_disp_sexo,calib_desv_sexo,eod_raza,fpr_disp_raza,calib_desv_raza,eod_ingreso,fpr_disp_ingreso,calib_desv_ingreso
0,xgboost,0,0.8271,0.0694,0.0116,0.0447,0.1305,0.0159,0.1385,0.1767,0.0334,0.0740
1,xgboost,1,0.8246,0.0649,0.0092,0.0593,0.1053,0.0131,0.2727,0.1694,0.0312,0.1227
2,xgboost,2,0.8280,0.0542,0.0093,0.0365,0.1114,0.0208,0.3424,0.1862,0.0358,0.1227
3,xgboost,3,0.8239,0.0618,0.0099,0.0504,0.1124,0.0176,0.4329,0.1704,0.0435,0.1044
4,xgboost,4,0.8268,0.0624,0.0085,0.0360,0.1087,0.0148,0.1485,0.1559,0.0400,0.0652


In [45]:
resumen_cv

,auroc,eod_sexo,fpr_disp_sexo,calib_desv_sexo,eod_raza,fpr_disp_raza,calib_desv_raza,eod_ingreso,fpr_disp_ingreso,calib_desv_ingreso
mean,0.8261,0.0625,0.0097,0.0454,0.1137,0.0164,0.2670,0.1717,0.0368,0.0978
std,0.0017,0.0056,0.0012,0.0098,0.0098,0.0029,0.1263,0.0111,0.0050,0.0270


## Guardado de resultados

Los resultados por pliegue se guardan en `results/metrics_xgboost.csv`, archivo que será consumido posteriormente por el cuaderno de comparación de los cuatro modelos.

In [46]:
import os

os.makedirs("../results", exist_ok=True)
tabla_metricas.to_csv("../results/metrics_xgboost.csv", index=False)
print("guardado en results/metrics_xgboost.csv")

guardado en results/metrics_xgboost.csv


## Detalle por grupo demográfico

Además del resumen anterior, se conserva el desglose por grupo (incluyendo el tamaño de muestra `n` de cada uno) para poder identificar grupos pequeños cuyas métricas sean menos estables, como algunas categorías de `_RACE` con pocos cientos de registros. Este detalle se guarda en `results/detalle_xgboost.csv`.

In [47]:
tabla_detalle = pd.concat(detalle_total, ignore_index=True)
tabla_detalle = tabla_detalle[["modelo", "pliegue", "atributo", "grupo", "n", "tpr", "fpr", "calibracion"]]

tabla_detalle.to_csv("../results/detalle_xgboost.csv", index=False)
print("guardado en results/detalle_xgboost.csv")

tabla_detalle.sort_values("n").head(10)

guardado en results/detalle_xgboost.csv


,modelo,pliegue,atributo,grupo,n,tpr,fpr,calibracion
9,xgboost,0,raza,5.0,381,0.152174,0.005970,0.861540
77,xgboost,4,raza,5.0,398,0.080000,0.008621,0.910807
60,xgboost,3,raza,5.0,404,0.156863,0.008499,0.881941
26,xgboost,1,raza,5.0,405,0.126984,0.005848,0.727281
43,xgboost,2,raza,5.0,422,0.117647,0.005155,1.342360
7,xgboost,0,raza,6.0,693,0.166667,0.008292,1.019161
40,xgboost,2,raza,6.0,723,0.150538,0.023810,1.068605
56,xgboost,3,raza,6.0,724,0.090000,0.020833,0.962813
72,xgboost,4,raza,6.0,739,0.099010,0.017241,0.978671
24,xgboost,1,raza,6.0,751,0.127660,0.010654,1.016974


## Comparación exploratoria: SMOTE vs. class weighting

Esta sección es exploratoria y no forma parte del resultado principal reportado arriba. Se repite el mismo ciclo de 5 pliegues, con los mismos hiperparámetros, pero sustituyendo SMOTE por `scale_pos_weight` (el equivalente de `class_weight="balanced"` en XGBoost), calculado en cada pliegue como la razón entre la clase mayoritaria y la minoritaria del train de ese pliegue. El objetivo es observar si una alternativa que no genera datos sintéticos produce resultados de equidad distintos a los de SMOTE.

In [48]:
resultados_cw = []

for i, (tr_idx, te_idx) in enumerate(folds):
    X_train, X_test = X.iloc[tr_idx].reset_index(drop=True), X.iloc[te_idx].reset_index(drop=True)
    y_train, y_test = y.iloc[tr_idx].reset_index(drop=True), y.iloc[te_idx].reset_index(drop=True)
    prot_test = protegidas.iloc[te_idx].reset_index(drop=True)

    X_train, X_test = pl.imputar_proporcional(X_train, X_test)
    X_train, X_test = pl.normalizar(X_train, X_test)
    # Sin SMOTE: el balanceo lo hace el modelo, no el dato.

    conteo_clases = y_train.value_counts()
    scale_pos_weight = conteo_clases[0] / conteo_clases[1]

    modelo_cw = XGBClassifier(
        **MEJORES_HIPERPARAMETROS,
        scale_pos_weight=scale_pos_weight,
        tree_method="hist",
        random_state=pl.RANDOM_STATE,
        eval_metric="logloss",
    )
    modelo_cw.fit(X_train, y_train)

    y_pred = modelo_cw.predict(X_test)
    y_proba = modelo_cw.predict_proba(X_test)[:, 1]

    resumen_cw, _ = pl.evaluar(y_test, y_pred, y_proba, prot_test)
    resumen_cw["modelo"] = "xgboost_class_weight"
    resumen_cw["pliegue"] = i
    resultados_cw.append(resumen_cw)
    print(f"pliegue {i} listo | AUROC = {resumen_cw['auroc']:.4f} | scale_pos_weight = {scale_pos_weight:.2f}")

pliegue 0 listo | AUROC = 0.8260 | scale_pos_weight = 7.46
pliegue 1 listo | AUROC = 0.8239 | scale_pos_weight = 7.46
pliegue 2 listo | AUROC = 0.8277 | scale_pos_weight = 7.46
pliegue 3 listo | AUROC = 0.8237 | scale_pos_weight = 7.46
pliegue 4 listo | AUROC = 0.8261 | scale_pos_weight = 7.46


In [49]:
tabla_cw = pd.DataFrame(resultados_cw)
tabla_cw = tabla_cw[columnas_ordenadas]

tabla_cw.to_csv("../results/metrics_xgboost_class_weight.csv", index=False)
print("guardado en results/metrics_xgboost_class_weight.csv")

comparacion = pd.DataFrame({
    "smote": tabla_metricas.drop(columns=["modelo", "pliegue"]).mean(),
    "class_weight": tabla_cw.drop(columns=["modelo", "pliegue"]).mean(),
}).round(4)
comparacion

guardado en results/metrics_xgboost_class_weight.csv


,smote,class_weight
auroc,0.8261,0.8255
eod_sexo,0.0625,0.1083
fpr_disp_sexo,0.0097,0.0726
calib_desv_sexo,0.0454,2.2705
eod_raza,0.1137,0.2741
fpr_disp_raza,0.0164,0.2205
calib_desv_raza,0.2670,3.1439
eod_ingreso,0.1717,0.3395
fpr_disp_ingreso,0.0368,0.3691
calib_desv_ingreso,0.0978,2.8532


El AUROC promedio es prácticamente igual entre ambas técnicas (SMOTE: 0.8261, class weighting: 0.8255), por lo que no hay diferencia relevante en el desempeño general. La diferencia aparece en las métricas de equidad: con `scale_pos_weight`, el EOD y la disparidad de FPR aumentan en los tres atributos protegidos respecto a SMOTE, y la desviación de calibración se dispara de forma notable (por ejemplo, de 0.27 a 3.14 en raza). Esto es consistente con lo señalado en la literatura citada en la Sección II-B del artículo: `scale_pos_weight` corrige el desbalance inflando uniformemente las probabilidades predichas para la clase minoritaria, lo cual empeora la calibración, mientras que SMOTE preserva una calibración considerablemente mejor al generar ejemplos sintéticos en vez de reponderar la función de pérdida. Con base en esto, se mantiene SMOTE como técnica principal para los resultados reportados en este cuaderno.